# Basic grid generation

This notebook shows how the grids that pylovo generated for a postcode area (PLZ) are stored in the database, how one grid is identified and how its pandapower network is read and plotted. It is the first of the grid-generation tutorials.

Grids are generated on the command line, not in this notebook:

```bash
pylovo-generate --plz 85653
```

The buildings, streets and transformers of the PLZ are loaded automatically; the settings are in `config/config_generation.yaml`.

**Prerequisites**

- pylovo installed with the `notebooks` extra (`uv sync --extra notebooks`) and this notebook running on the kernel of the project environment (`.venv`), see `notebook_tutorials/README.md`.
- A `.env` in the repository root that points to your pylovo database, a completed `pylovo-setup` and generated grids for the postcode area (PLZ) below, e.g. `pylovo-generate --plz 85653`.
- Grids are read for the `VERSION_ID` of `config/config_generation.yaml`.

The stored outputs were generated for the demo postcode area 85653 (Aying), whose buildings, streets and transformers are derived from OpenStreetMap. Map data © OpenStreetMap contributors (ODbL).

In [1]:
# Parameters
plz = 85653  # postcode area with generated grids
kcid = None  # k-means cluster id of the grid; None: first grid of the PLZ
bcid = None  # building cluster id of the grid; None: first grid of the PLZ

## Setup
Connect to the database and select a grid.

In [2]:
import warnings

import plotly.io as pio

from pylovo.config_loader import VERSION_ID
from pylovo.database.database_client import DatabaseClient
from pylovo.plotting.generation.networks import plot_simple_grid

warnings.filterwarnings("ignore")
# pandapower's plotly functions select the "notebook" renderer, which embeds the complete
# plotly.js (about 4.7 MB) for every figure. The CDN variant keeps this notebook small.
pio.renderers["notebook"] = pio.renderers["notebook_connected"]


### Grids of the PLZ
A PLZ contains several low-voltage grids. A grid is identified by the `VERSION_ID` of the configuration, the `plz` (postcode area), the `kcid` (k-means cluster id) and the `bcid` (building cluster id). The cluster list holds the `(kcid, bcid)` tuples of all grids of the PLZ. A negative `bcid` means that the grid is supplied by an existing transformer (OSM or DSO data); grids with a non-negative `bcid` got a newly placed transformer.

In [3]:
dbc = DatabaseClient()  # read-only access (GridGenerator is only needed to generate grids)
cluster_list = dbc.get_list_from_plz(plz)
if not cluster_list:
    raise ValueError(f"No grids of version {VERSION_ID} for PLZ {plz}. Run: pylovo-generate --plz {plz}")
if kcid is None or bcid is None:
    kcid, bcid = cluster_list[0]
print(f"PLZ {plz}, version {VERSION_ID}: {len(cluster_list)} grids; selected kcid={kcid}, bcid={bcid}")

PLZ 85653, version 1: 4 grids; selected kcid=1, bcid=1


In [4]:
cluster_list

[(1, 1), (1, 2), (1, 3), (1, 4)]

### Investigate one grid
The pandapower network of the selected grid is read from the database:

In [5]:
net = dbc.read_net_db(plz=plz, kcid=kcid, bcid=bcid)
net

This pandapower network includes the following parameter tables:
   - bus (283 elements)
   - load (144 elements)
   - ext_grid (1 element)
   - line (281 elements)
   - trafo (1 element)
 and the following results tables:
   - res_bus (283 elements)
   - res_line (281 elements)
   - res_trafo (1 element)
   - res_ext_grid (1 element)
   - res_load (144 elements)

The grid with its geographic coordinates (hover over the buses and lines for details). The MV bus and the LV bus of the transformer lie on top of each other.

In [6]:
plot_simple_grid(plz=plz, kcid=kcid, bcid=bcid)

The buses shown above, identified by their index. With pandapower 3 the coordinates are stored as GeoJSON (WGS84) in the `geo` column of the element tables.

In [7]:
net.bus[["name", "vn_kv", "zone", "geo"]].head(10)

,name,vn_kv,zone,geo
0,LVbus 1,0.4,n,"{""coordinates"":[11.773804221522283,47.96867261..."
1,MVbus 1,20.0,n,"{""coordinates"":[11.773804221522283,47.96882261..."
2,Connection Nodebus 377,0.4,n,"{""coordinates"":[11.773804221522283,47.96867261..."
3,Connection Nodebus 372,0.4,n,"{""coordinates"":[11.773780531608997,47.96875837..."
4,Connection Nodebus 380,0.4,n,"{""coordinates"":[11.773847258420133,47.96851682..."
5,Connection Nodebus 382,0.4,n,"{""coordinates"":[11.773860679780046,47.96846823..."
6,Connection Nodebus 386,0.4,n,"{""coordinates"":[11.773969933750914,47.96895798..."
7,Connection Nodebus 385,0.4,n,"{""coordinates"":[11.773926677678574,47.96822934..."
8,Connection Nodebus 346,0.4,n,"{""coordinates"":[11.773671095203593,47.96915760..."
9,Connection Nodebus 324,0.4,n,"{""coordinates"":[11.773383244819772,47.96908744..."


The following notebooks explain the generated grids in more detail: `1_map_visualization_examples.ipynb` builds up one grid step by step on a map.